# Kaggle – Data on the top
Tu profe ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Métrica: RMSE

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

Donde $y_i$ es el valor real y $\hat{y}_i$ es el valor predicho. **Cuanto menor, mejor.**

---
# PARTE 1: Entrenamiento del modelo

## 1. Librerías

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
import re


## 2. Datos

In [2]:
train = pd.read_csv('./data/train.csv', encoding='latin-1')
test = pd.read_csv("./data/test.csv")


### 2.1 Exploración de los datos

In [3]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 912 entries, 0 to 911
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   laptop_ID         912 non-null    int64  
 1   Company           912 non-null    str    
 2   Product           912 non-null    str    
 3   TypeName          912 non-null    str    
 4   Inches            912 non-null    float64
 5   ScreenResolution  912 non-null    str    
 6   Cpu               912 non-null    str    
 7   Ram               912 non-null    str    
 8   Memory            912 non-null    str    
 9   Gpu               912 non-null    str    
 10  OpSys             912 non-null    str    
 11  Weight            912 non-null    str    
 12  Price_in_euros    912 non-null    float64
dtypes: float64(2), int64(1), str(10)
memory usage: 203.7 KB


In [4]:
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 391 entries, 0 to 390
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   laptop_ID         391 non-null    int64  
 1   Company           391 non-null    str    
 2   Product           391 non-null    str    
 3   TypeName          391 non-null    str    
 4   Inches            391 non-null    float64
 5   ScreenResolution  391 non-null    str    
 6   Cpu               391 non-null    str    
 7   Ram               391 non-null    str    
 8   Memory            391 non-null    str    
 9   Gpu               391 non-null    str    
 10  OpSys             391 non-null    str    
 11  Weight            391 non-null    str    
dtypes: float64(1), int64(1), str(10)
memory usage: 84.5 KB


In [5]:
train.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
0,755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.86kg,539.00
1,618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01
2,909,HP,ProBook 450,Notebook,15.6,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8GB,1TB HDD,Nvidia GeForce 930MX,Windows 10,2.04kg,900.00
3,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
4,286,Dell,Inspiron 3567,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,AMD Radeon R5 M430,Linux,2.25kg,428.00


In [6]:
def describe_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Genera un resumen estadístico descriptivo de un DataFrame.

    Argumentos:
        df (pd.DataFrame): DataFrame a analizar.

    Retorna:
        pd.DataFrame: DataFrame con una fila por columna del input y las
        siguientes columnas: 'tipo', 'porcentaje_nulos', 'valores_unicos',
        'porcentaje_cardinalidad'.
        Retorna None si el input no es un DataFrame válido.
    """
    
    # Comprobación de que si es un DataFrame
    if not isinstance(df, pd.DataFrame):
        print("Error: el objeto proporcionado no es un DataFrame.")
        return None

    # Crear el DataFrame resultado
    resultado = pd.DataFrame(index=df.columns)

    # Tipo de dato
    resultado["tipo"] = df.dtypes.astype(str)

    # Porcentaje de nulos
    resultado["porcentaje_nulos"] = (df.isna().mean() * 100).round(2)

    # Valores únicos
    resultado["valores_unicos"] = df.nunique()

    # Porcentaje de cardinalidad
    resultado["porcentaje_cardinalidad"] = ((df.nunique() / len(df)) * 100).round(2)


    return resultado

In [7]:
describe_df(train)

,tipo,porcentaje_nulos,valores_unicos,porcentaje_cardinalidad
laptop_ID,int64,0.0,912,100.00
Company,str,0.0,19,2.08
Product,str,0.0,480,52.63
TypeName,str,0.0,6,0.66
Inches,float64,0.0,17,1.86
ScreenResolution,str,0.0,36,3.95
Cpu,str,0.0,107,11.73
Ram,str,0.0,9,0.99
Memory,str,0.0,37,4.06
Gpu,str,0.0,93,10.20


In [8]:
train["Cpu"].value_counts()

Cpu
Intel Core i5 7200U 2.5GHz              124
Intel Core i7 7700HQ 2.8GHz             105
Intel Core i7 7500U 2.7GHz               97
Intel Core i5 8250U 1.6GHz               52
Intel Core i7 8550U 1.8GHz               47
                                       ... 
Intel Core i3 6006U 2.2GHz                1
Intel Atom Z8350 1.92GHz                  1
Intel Core i5 7200U 2.50GHz               1
AMD A6-Series 7310 2GHz                   1
Intel Pentium Dual Core N4200 1.1GHz      1
Name: count, Length: 107, dtype: int64

In [9]:
def process_cpu(df):
    # Fabricante
    df["CPU_Brand"] = df["Cpu"].str.extract(r'^(Intel|AMD)', expand=False)

    # Familia
    df["CPU_Family"] = df["Cpu"].str.extract(
        r'(Core i[3579]|Celeron|Pentium|Atom|Xeon|Ryzen [3579]|A\d-Series|FX)',
        expand=False
    )

    # Frecuencia (GHz)
    df["CPU_GHz"] = (
        df["Cpu"]
        .str.extract(r'(\d+\.?\d*)\s*GHz', expand=False)
        .astype(float)
    )

    # Número de modelo (ej. 7200, 8550, 2700...)
    df["CPU_Model"] = df["Cpu"].str.extract(r'(\d{4})', expand=False)

    # Generación Intel (7200 -> 7, 8550 -> 8)
    df["CPU_Generation"] = df["CPU_Model"].str[0]

    # Sufijo (U, HQ, HK, H...)
    df["CPU_Suffix"] = df["Cpu"].str.extract(r'(\d{4})([A-Z]{1,3})', expand=True)[1]

    return df

In [10]:
# Todo lo que hacemos a train se lo hacemos a test
process_cpu(train)
process_cpu(test)
train.head(2)


,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros,CPU_Brand,CPU_Family,CPU_GHz,CPU_Model,CPU_Generation,CPU_Suffix
0,755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.86kg,539.00,Intel,Core i3,2.0,6006,6,U
1,618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01,Intel,Core i7,2.6,6700,6,HQ


In [11]:
train["ScreenResolution"].value_counts()

ScreenResolution
Full HD 1920x1080                                349
1366x768                                         211
IPS Panel Full HD 1920x1080                      163
IPS Panel Full HD / Touchscreen 1920x1080         32
Full HD / Touchscreen 1920x1080                   30
1600x900                                          14
Touchscreen 1366x768                              11
Quad HD+ / Touchscreen 3200x1800                  11
IPS Panel 4K Ultra HD / Touchscreen 3840x2160     10
4K Ultra HD / Touchscreen 3840x2160                7
IPS Panel Quad HD+ / Touchscreen 3200x1800         6
Touchscreen 2560x1440                              6
IPS Panel 4K Ultra HD 3840x2160                    5
IPS Panel Retina Display 2560x1600                 5
Touchscreen 2256x1504                              5
1440x900                                           4
IPS Panel 1366x768                                 4
IPS Panel Retina Display 2304x1440                 4
IPS Panel Touchscreen 2560x14

In [12]:
def process_screen(df):

    # Touchscreen
    df["Touchscreen"] = df["ScreenResolution"].str.contains("Touchscreen").astype(int)

    # IPS
    df["IPS"] = df["ScreenResolution"].str.contains("IPS").astype(int)

    # Retina
    df["Retina"] = df["ScreenResolution"].str.contains("Retina").astype(int)

    # Resolución
    resolution = df["ScreenResolution"].str.extract(r'(\d+)x(\d+)')

    df["Resolution_X"] = resolution[0].astype(int)
    df["Resolution_Y"] = resolution[1].astype(int)

    # Número de píxeles
    df["Pixels"] = df["Resolution_X"] * df["Resolution_Y"]

    # Relación de aspecto
    df["Aspect_Ratio"] = df["Resolution_X"] / df["Resolution_Y"]

    return df

In [13]:
process_screen(train)
process_screen(test)
train.head(2)

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,...,CPU_Model,CPU_Generation,CPU_Suffix,Touchscreen,IPS,Retina,Resolution_X,Resolution_Y,Pixels,Aspect_Ratio
0,755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,...,6006,6,U,0,0,0,1920,1080,2073600,1.777778
1,618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,...,6700,6,HQ,0,0,0,1920,1080,2073600,1.777778


In [14]:
train["Memory"].value_counts()

Memory
256GB SSD                        282
1TB HDD                          152
500GB HDD                         92
512GB SSD                         83
128GB SSD +  1TB HDD              67
128GB SSD                         54
256GB SSD +  1TB HDD              52
32GB Flash Storage                33
1TB SSD                           12
64GB Flash Storage                11
2TB HDD                            8
512GB SSD +  1TB HDD               8
256GB Flash Storage                7
16GB Flash Storage                 6
256GB SSD +  2TB HDD               6
1.0TB Hybrid                       5
32GB SSD                           5
128GB Flash Storage                4
180GB SSD                          3
16GB SSD                           3
1TB SSD +  1TB HDD                 2
512GB SSD +  2TB HDD               2
256GB SSD +  256GB SSD             1
128GB SSD +  2TB HDD               1
512GB SSD +  512GB SSD             1
512GB SSD +  256GB SSD             1
64GB Flash Storage +  1TB HDD  

In [15]:
def process_memory(df):

    df["SSD"] = 0
    df["HDD"] = 0
    df["Flash"] = 0
    df["Hybrid"] = 0

    for i, mem in enumerate(df["Memory"]):

        if pd.isna(mem):
            continue

        devices = mem.split("+")

        for d in devices:

            d = d.strip()

            size = re.search(r'(\d+\.?\d*)(GB|TB)', d)

            if size:

                value = float(size.group(1))
                unit = size.group(2)

                if unit == "TB":
                    value *= 1024

                if "SSD" in d:
                    df.at[i, "SSD"] += value

                elif "HDD" in d:
                    df.at[i, "HDD"] += value

                elif "Flash Storage" in d:
                    df.at[i, "Flash"] += value

                elif "Hybrid" in d:
                    df.at[i, "Hybrid"] += value

    df["Total_Storage"] = (
        df["SSD"] +
        df["HDD"] +
        df["Flash"] +
        df["Hybrid"]
    )

    return df

In [16]:
process_memory(train)
process_memory(test)
train.head(2)

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,...,Retina,Resolution_X,Resolution_Y,Pixels,Aspect_Ratio,SSD,HDD,Flash,Hybrid,Total_Storage
0,755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,...,0,1920,1080,2073600,1.777778,256,0,0,0,256
1,618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,...,0,1920,1080,2073600,1.777778,0,1024,0,0,1024


In [17]:
# Quitamos el GB de la columna Ram para dejarla en un valor numérico
train['Ram'] = train['Ram'].str.extract('(\d+)').astype(int)
test['Ram'] = test['Ram'].str.extract('(\d+)').astype(int)

# Quitamos el kg de la columna Weight para dejarla en un valor numérico
train['Weight'] = train['Weight'].str.replace('kg', '', regex=False).str.strip().astype(float)
test['Weight'] = test['Weight'].str.replace('kg', '', regex=False).str.strip().astype(float)

<>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\Usuario\AppData\Local\Temp\ipykernel_22156\3561433026.py:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  train['Ram'] = train['Ram'].str.extract('(\d+)').astype(int)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_22156\3561433026.py:3: SyntaxWarning: "\d" is an invalid escape

In [18]:
train["Gpu"].value_counts()

Gpu
Intel HD Graphics 620       185
Intel HD Graphics 520       125
Intel UHD Graphics 620       52
Nvidia GeForce GTX 1050      48
Nvidia GeForce 940MX         31
                           ... 
AMD Radeon Pro 555            1
Nvidia Quadro M2000M          1
Nvidia GeForce GTX 940M       1
AMD Radeon R5 520             1
Nvidia GeForce GTX 1070M      1
Name: count, Length: 93, dtype: int64

In [19]:
def process_gpu(df):

    # Fabricante
    df["GPU_Brand"] = df["Gpu"].str.extract(
        r'^(Intel|AMD|Nvidia)',
        expand=False
    )

    # Familia
    df["GPU_Family"] = df["Gpu"].str.extract(
        r'(GeForce|Quadro|Radeon|FirePro|Iris|HD Graphics|UHD Graphics)',
        expand=False
    )

    # Número de modelo
    df["GPU_Model"] = df["Gpu"].str.extract(
        r'(\d{3,4})',
        expand=False
    )

    # GTX
    df["GTX"] = df["Gpu"].str.contains("GTX", case=False).astype(int)

    # RTX (por si aparecen en otros datasets)
    df["RTX"] = df["Gpu"].str.contains("RTX", case=False).astype(int)

    # MX
    df["MX"] = df["Gpu"].str.contains("MX", case=False).astype(int)

    return df

In [20]:
process_gpu(train)
process_gpu(test)
train.head(2)

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,...,HDD,Flash,Hybrid,Total_Storage,GPU_Brand,GPU_Family,GPU_Model,GTX,RTX,MX
0,755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8,256GB SSD,Intel HD Graphics 520,...,0,0,0,256,Intel,HD Graphics,520,0,0,0
1,618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16,1TB HDD,Nvidia GeForce GTX 960<U+039C>,...,1024,0,0,1024,Nvidia,GeForce,960,1,0,0


In [21]:
train = train.drop(columns=['laptop_ID', 'Product', 'Cpu', 'Gpu','ScreenResolution', 'Memory'])
test = test.drop(columns=['laptop_ID', 'Product', 'Cpu', 'Gpu','ScreenResolution', 'Memory'])

In [22]:
describe_df(train)

,tipo,porcentaje_nulos,valores_unicos,porcentaje_cardinalidad
Company,str,0.00,19,2.08
TypeName,str,0.00,6,0.66
Inches,float64,0.00,17,1.86
Ram,int64,0.00,9,0.99
OpSys,str,0.00,9,0.99
Weight,float64,0.00,158,17.32
Price_in_euros,float64,0.00,603,66.12
CPU_Brand,str,0.00,2,0.22
CPU_Family,str,3.73,11,1.21
CPU_GHz,float64,0.00,25,2.74


In [23]:

cat_cols = [
    "Company",
    "TypeName",
    "OpSys",
    "CPU_Brand",
    "CPU_Family",
    "CPU_Generation",
    "CPU_Suffix",
    "GPU_Brand",
    "GPU_Family",
    "CPU_Model",
    "GPU_Model"
]

for col in cat_cols:
    train[col] = train[col].fillna("Unknown").astype(str)
    test[col] = test[col].fillna("Unknown").astype(str)




In [24]:
describe_df(train)

,tipo,porcentaje_nulos,valores_unicos,porcentaje_cardinalidad
Company,str,0.0,19,2.08
TypeName,str,0.0,6,0.66
Inches,float64,0.0,17,1.86
Ram,int64,0.0,9,0.99
OpSys,str,0.0,9,0.99
Weight,float64,0.0,158,17.32
Price_in_euros,float64,0.0,603,66.12
CPU_Brand,str,0.0,2,0.22
CPU_Family,str,0.0,12,1.32
CPU_GHz,float64,0.0,25,2.74


In [25]:
# Eliminamos por que tiene un solo valor único (no aporta nada)
train = train.drop(columns="RTX")
test = test.drop(columns="RTX")

### 2.2 Definir X e y


### 2.3 Dividir en train y test

In [26]:
from sklearn.model_selection import train_test_split

X = train.drop("Price_in_euros", axis=1)
y = train["Price_in_euros"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## 3. Procesado de datos

> 🚨 **Data leakage:** si usas un scaler, haz **`.fit()` SOLO sobre `X_train`** y luego aplica `.transform()` sobre `X_train` y `X_test` por separado.
>
> Recuerda también que **todo lo que hagas aquí deberás replicarlo después en `test.csv`** (sección 6).

In [50]:
# como voy a utilizar el catboos no necesita ni escalado ni one hot encoder de variables cat

## 4. Modelado

In [27]:
from catboost import CatBoostRegressor

model = CatBoostRegressor(
    iterations=100,
    learning_rate=0.1,
    depth=3,
    loss_function="RMSE",
    random_seed=42,
    verbose=False
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_cols
)

CatBoostRegressor(depth=3, iterations=100, learning_rate=0.1, loss_function='RMSE', random_seed=42, verbose=False)

In [28]:
from sklearn.metrics import root_mean_squared_error

pred = model.predict(X_val)

rmse = root_mean_squared_error(y_val, pred)

print("RMSE:", rmse)

RMSE: 317.37259114865486


In [29]:
# Importancia de las variables

importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

print(importance.head(15))

         Feature  Importance
3            Ram   29.023938
1       TypeName   15.016691
19           SSD    9.014398
5         Weight    8.627782
7     CPU_Family    7.334753
16  Resolution_Y    6.747892
17        Pixels    5.085490
8        CPU_GHz    4.161930
11    CPU_Suffix    2.891342
26     GPU_Model    2.819978
2         Inches    2.667408
4          OpSys    2.276051
25    GPU_Family    1.730318
15  Resolution_X    1.247305
20           HDD    0.618405


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor

model = CatBoostRegressor(
    loss_function="RMSE",
    random_seed=42,
    verbose=0
)

param_dist = {
    "depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1],
    "iterations": [300, 500, 800],
    "l2_leaf_reg": [1, 3, 5],
    "bagging_temperature": [0, 1, 3],
    "random_strength": [1, 2, 5],
    "grow_policy": ["SymmetricTree"],
    "subsample": [0.7, 1.0]
}

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=15,
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

search.fit(
    X_train,
    y_train,
    cat_features=cat_cols
)

print(search.best_params_)
print(-search.best_score_)


Fitting 3 folds for each of 15 candidates, totalling 45 fits


In [59]:
best_model = search.best_estimator_

y_val_pred = best_model.predict(X_val)

rmse_val = root_mean_squared_error(y_val, y_val_pred)
print("RMSE validación:", rmse_val)


RMSE validación: 284.3599435201122


### 4.3 Optimización (up to you 🫰🏻)

## 5. Reentrenamiento sobre todos los datos de `train.csv`

Una vez afinado el modelo, reentrenamos con **todos** los datos disponibles antes de predecir sobre `test.csv`.

> ¿Por qué? El split anterior era solo para validar localmente. Para la submission final queremos aprovechar el 100% de los datos de entrenamiento.

In [60]:
# Unir train y val para entrenar el modelo final
X_train_full = pd.concat([X_train, X_val], axis=0)
y_train_full = pd.concat([y_train, y_val], axis=0)


In [61]:
# Entrenar el modelo final con todos los datos
best_model.fit(
    X_train_full,
    y_train_full,
    cat_features=cat_cols
)


CatBoostRegressor(bagging_temperature=3, depth=6, grow_policy='SymmetricTree', iterations=800, l2_leaf_reg=1, learning_rate=0.05, loss_function='RMSE', random_seed=42, random_strength=2, subsample=0.7, verbose=0)

---
# PARTE 2: Predicción y submission

Una vez tengas el modelo listo, toca predecir sobre `test.csv` y generar el archivo de submission.

## 6. Carga los datos de `test.csv`

## 7. Replica el procesado en `test.csv`

> ⚠️ Usa `.transform()`, **nunca `.fit_transform()`** sobre los datos de test.
>
> Lo único que **no puedes hacer** es eliminar filas.

## 8. Genera la submission

### 8.1 ¿Qué formato espera Kaggle?

In [64]:
sample = pd.read_csv('./data/sample_submission.csv', encoding='latin-1')
sample.head()

,laptop_ID,Price_in_euros
0,209,1949.1
1,1281,805.0
2,1168,1101.0
3,1231,1293.8
4,1020,1832.6


### 8.2 Crea tu submission

In [65]:
# Predicciones sobre el test
predictions_submit = best_model.predict(test)

# Crear submission
submission = pd.DataFrame({
    'laptop_ID': sample['laptop_ID'],
    'Price_in_euros': predictions_submit
})

submission.head()

,laptop_ID,Price_in_euros
0,209,1631.467790
1,1281,303.378371
2,1168,319.494549
3,1231,1102.043496
4,1020,1031.519828


### 8.3 Chequeador

Pásale el chequeador antes de subir a Kaggle. Si todo está bien, guardará el CSV automáticamente con un nombre único.

In [66]:
def checker(df_to_submit, sample, filename=None):
    """
    Valida que tu submission tenga la forma requerida por Kaggle.
    Si es correcta, guarda el CSV listo para subir.
    Si no, lee el mensaje de error y corrígelo.
    """
    if df_to_submit.shape != sample.shape:
        print(' Shape incorrecto.')
        print(f'   Tu submission: {df_to_submit.shape} | Esperado: {sample.shape}')
        print('   Revisa que no hayas borrado filas del test ni añadido/quitado columnas.')
        return

    if not (df_to_submit.columns == sample.columns).all():
        print(' Nombres de columnas incorrectos.')
        print(f'   Tus columnas:       {list(df_to_submit.columns)}')
        print(f'   Columnas esperadas: {list(sample.columns)}')
        return

    if not (df_to_submit['laptop_ID'] == sample['laptop_ID']).all():
        print(' Los IDs no coinciden con sample_submission. Revisa que no hayas reordenado el test.csv.')
        return

    if filename is None:
        from datetime import datetime
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f'submission_{timestamp}.csv'

    df_to_submit.to_csv(filename, index=False)
    print(f" ¡Todo correcto! Submission guardada como '{filename}'. ¡A Kaggle!")

In [67]:
checker(submission, sample)

 ¡Todo correcto! Submission guardada como 'submission_20260701_002258.csv'. ¡A Kaggle!


In [ ]:
# mejorar el rango de hiperparámetros

In [69]:
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor

model = CatBoostRegressor(
    loss_function="RMSE",
    random_seed=42,
    verbose=0
)

param_dist = {
    "depth": [4, 5, 6, 7, 8],
    "learning_rate": [0.02, 0.03, 0.04, 0.05],
    "iterations": [600, 800, 1000, 1200],
    "l2_leaf_reg": [1, 3, 5, 7, 10],
    "bagging_temperature": [0, 1, 3, 5, 10],
    "random_strength": [1, 2, 3, 5],
    "grow_policy": ["SymmetricTree"],
    "subsample": [0.7, 0.8, 0.9, 1.0]
}


search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=40,   # más combinaciones → mejor
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

search.fit(
    X_train,
    y_train,
    cat_features=cat_cols
)

print(search.best_params_)
print(-search.best_score_)


Fitting 3 folds for each of 40 candidates, totalling 120 fits
{'subsample': 0.7, 'random_strength': 2, 'learning_rate': 0.05, 'l2_leaf_reg': 5, 'iterations': 1200, 'grow_policy': 'SymmetricTree', 'depth': 5, 'bagging_temperature': 10}
272.81001226347706


In [70]:
best_model_2 = search.best_estimator_

y_val_pred = best_model_2.predict(X_val)
rmse_val = root_mean_squared_error(y_val, y_val_pred)
print("RMSE validación:", rmse_val)


RMSE validación: 262.01602776955053


In [71]:
# Predicciones sobre el test
predictions_submit_2 = best_model_2.predict(test)

# Crear submission
submission_2 = pd.DataFrame({
    'laptop_ID': sample['laptop_ID'],
    'Price_in_euros': predictions_submit_2
})

submission_2.head()

,laptop_ID,Price_in_euros
0,209,1690.485727
1,1281,267.804237
2,1168,380.574315
3,1231,1031.691134
4,1020,930.235948


In [72]:
checker(submission_2, sample)

 ¡Todo correcto! Submission guardada como 'submission_20260701_005026.csv'. ¡A Kaggle!


In [ ]:
param_dist = {
    "depth": [6, 7, 8],                     # más capacidad sin ser lento
    "learning_rate": [0.025, 0.03, 0.035],  # LR bajo = mejor test
    "iterations": [900, 1200, 1500],        # suficiente para LR bajo
    "l2_leaf_reg": [3, 5, 7, 10],           # controla overfitting
    "bagging_temperature": [0, 1, 3],       # robustez sin ralentizar
    "random_strength": [1, 2, 3],           # diversidad de árboles
    "subsample": [0.7, 0.8, 0.9],           # reduce varianza
    "grow_policy": ["SymmetricTree"]        # estable y rápido
}
